# DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic en Kaggle

Modelo: [xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic](https://huggingface.co/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic) (9B params, BF16, ~18 GB, multimodal).

**Antes de correr:**
- Settings (panel derecho) → **Accelerator: GPU T4 x2**
- Settings → **Internet: ON**
- La primera carga descarga ~18 GB (10-15 min).
- La prueba de imagen usa imagen reducida + tope de píxeles para no agotar la memoria de las T4 (no tienen flash-attention).

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U transformers accelerate

In [ ]:
import torch
print(f"{torch.cuda.device_count()} GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

MODEL = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"

processor = AutoProcessor.from_pretrained(MODEL)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL,
    device_map="auto",          # reparte el modelo entre las 2 T4
    torch_dtype="bfloat16",
)
print("Capas asignadas:", model.hf_device_map)

## Prueba de chat (texto)

In [ ]:
messages = [
    {"role": "user", "content": "Explícame en 3 frases qué es la destilación de modelos."}
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

out = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.7)
respuesta = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
print(respuesta)

## Prueba con imagen (imagen reducida para no agotar memoria)

Las T4 no tienen flash-attention: una imagen a máxima resolución genera miles de tokens visuales y la matriz de atención del encoder visual explota (~133 GB). Con el tope de píxeles y un thumbnail de 512px quedan unos cientos de tokens.

In [ ]:
import gc, requests
from io import BytesIO
from PIL import Image

gc.collect(); torch.cuda.empty_cache()

# Tope de píxeles => pocos cientos de tokens visuales en vez de miles
processor = AutoProcessor.from_pretrained(MODEL, min_pixels=224*224, max_pixels=448*448)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"
img = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
img.thumbnail((512, 512))   # máx 512px por lado

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "¿Qué animal hay en la foto?"},
        ],
    }
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

out = model.generate(**inputs, max_new_tokens=256)
print(processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

## Plan B: cuantización 4-bit (si te quedas sin memoria en los pesos)

Libera el modelo actual y recárgalo en 4-bit (~6 GB, entra en una sola T4).

In [ ]:
#!pip install -q bitsandbytes
#
#import gc
#del model, inputs, out
#gc.collect(); torch.cuda.empty_cache()
#
#from transformers import BitsAndBytesConfig
#
#model = AutoModelForMultimodalLM.from_pretrained(
#    MODEL,
#    device_map="auto",
#    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
#)